# Evaluating an ADK Agent in Agent Runtime with EvalBench

This notebook guides you through the process of:
1. Writing a minimal SQL-generating agent using **Agent Development Kit (ADK)**.
2. Deploying it to **Agent Runtime (Gemini Enterprise Agent Platform)** using the `AdkApp` template.
3. Querying the live endpoint to verify it streams valid SQL.
4. Evaluating the agent in Agent Runtime.

## Step 1: Repository Setup

In [1]:
# Install/Upgrade the Google Cloud AI Platform SDK, ADK, and authentication libraries
!pip install --quiet --upgrade google-cloud-aiplatform google-adk google-auth==2.49.0 pyaml_env

If running in a hosted environment like Colab Enterprise, we clone the `evalbench` repository to load evaluation datasets, model configurations, and pipeline execution shell scripts.

In [ ]:
!git clone https://github.com/GoogleCloudPlatform/evalbench.git

We install uv to manage python packages, sync the virtual environment, install the mcp library package, and compile the protobuf message files.

In [ ]:
# Re-install dependencies and compile protobufs
!pip install --quiet uv
!cd /content/evalbench && uv sync
!cd /content/evalbench && .venv/bin/python3 -m grpc_tools.protoc \
    --proto_path=evalbench/evalproto \
    --python_out=evalbench/evalproto \
    --pyi_out=evalbench/evalproto \
    --grpc_python_out=evalbench/evalproto \
    --experimental_editions evalbench/evalproto/*.proto

import os
os.chdir("/content/evalbench/docs/examples")
print(f"✅ Working directory set to: {os.getcwd()}")

Download the BIRD Dataset and Database Connections:
This script downloads the natural language questions, database schemas, and SQLite database connection files required to evaluate the SQL generator.

In [ ]:
# Download the BIRD dataset and database connections
!cd ../.. && bash datasets/bird/download_dataset.sh

## Step 2: Define the ADK Agent

We write the ADK agent definition code to a local file named `minimal_agent.py` using standard ADK syntax. We specify `gemini-2.5-flash` as the model name and provide custom instruction formatting rules.

In [ ]:
%%writefile minimal_agent.py
import google.adk as adk

def create_agent() -> adk.Agent:
    """Creates a minimal SQL-generating ADK agent."""
    return adk.Agent(
        name="minimal_adk_sql_agent",
        model="gemini-2.5-flash",
        instruction=(
            "You are a SQL generator. Generate a SQLite SQL query based on "
            "the user prompt. Wrap your final generated SQL query inside a "
            "```sql code block."
        )
    )

## Step 3: Configure and initialize Agent Platform client

Set your target Google Cloud Project and region, and initialize the SDK.

In [ ]:
from google.colab import auth
auth.authenticate_user()

Set your target Google Cloud Project and region, and initialize the SDK.

In [ ]:
import os
import vertexai
import google.auth

# Edit these variables with your Google Cloud details.
PROJECT_ID = "YOUR-PROJECT-ID"
LOCATION = "us-central1"
STAGING_BUCKET = f"gs://{PROJECT_ID}-agents"

# Get the Colab authenticated credentials.
print(f"Initializing global context and client...")
credentials, _ = google.auth.default()

# Initialize global context and client with explicit credentials.
vertexai.init(project=PROJECT_ID, location=LOCATION, credentials=credentials)
client = (
    vertexai.Client(
        project=PROJECT_ID, 
        location=LOCATION, 
        credentials=credentials
    )
)

# Export to environment variables for subprocesses (like evalbench.py)
os.environ["EVAL_GCP_PROJECT_ID"] = PROJECT_ID
os.environ["EVAL_GCP_PROJECT_REGION"] = LOCATION

## Step 4: Deploy to Agent Runtime using the AdkApp Template

We load the local ADK agent and wrap it inside the `AdkApp` class. Then, we call `client.agent_engines.create` to deploy the agent.

Since our agent is self-contained in `minimal_agent.py`, we only need to include `./minimal_agent.py` in the `extra_packages` deployment configuration.

In [ ]:
from vertexai.agent_engines.templates.adk import AdkApp

import minimal_agent


# Instantiate the local ADK agent and wrap it in the template.
local_agent = minimal_agent.create_agent()
adk_app = AdkApp(agent=local_agent, app_name="minimal-adk-sql-app")

print("Deploying minimal ADK agent to Agent Runtime...")
remote_agent = client.agent_engines.create(
    agent=adk_app,
    config=dict(
        display_name="minimal_adk_agent",
        staging_bucket=STAGING_BUCKET,
        requirements=[
            "google-cloud-aiplatform[agent_engines,adk]",
            "google-genai",
            "google-adk[all]==2.1.0"
        ],
        extra_packages=["./minimal_agent.py"]
    )
)

print(f"\n✅ Agent deployed successfully!")
print(f"Resource Name: {remote_agent.api_resource.name}")

## Step 5: Verify and test the live agent

Query your live deployed agent over streaming to verify it returns SQL wrapped inside markdown code blocks.

Once verified, copy the `Resource Name` printed in the previous step and paste it into the `resource_name` configuration field of your EvalBench `model_config` YAML file (e.g. `datasets/model_configs/agent_runtime.yaml`) to run evaluations!

In [ ]:
# Test query the remote app client over streaming.
print("Querying remote agent endpoint...")
response_stream = remote_agent.stream_query(
    message="How many schools are in California?",
    user_id="evalbench_test"
)

print("\nStreaming response chunks:")
for chunk in response_stream:
    print(chunk, end="")

## Step 6: Generate temporary config files for evaluation

To run evaluations without modifying the default files in the repository, we create temporary copies of both the model configuration and the run configuration.

The code block below:
1. Loads the default `agent_runtime.yaml` model config, updates the `resource_name` with your deployed agent ID, and writes it to a new temporary file `agent_runtime_temp.yaml`.
2. Loads the default run configuration, updates its `model_config` path to point to the temporary model config, and writes it to a new temporary file `test_agent_runtime_run_config_temp.yaml`.

In [ ]:
import os
import json
import yaml
import pyaml_env

# Using absolute paths to ensure files are located correctly
base_path = "/content/evalbench"
model_config_path = (
    os.path.join(base_path, "datasets/model_configs/agent_runtime.yaml")
)
prompts_path = os.path.join(base_path, "datasets/bird/prompts.json")

# Temporary file paths
temp_model_config_path = (
    os.path.join(base_path, "datasets/model_configs/agent_runtime_temp.yaml")
)
temp_prompts_path = (
    os.path.join(base_path, "datasets/bird/prompts_california_temp.json")
)
temp_run_config_path = (
    os.path.join(
        base_path, 
        "datasets/bird/test_agent_runtime_run_config_temp.yaml"
    )
)

# Load, update, and write temporary model config using pyaml_env to resolve !ENV tags.
model_data = pyaml_env.parse_config(model_config_path)

# Update the resource name with your deployed agent
model_data["resource_name"] = remote_agent.api_resource.name

with open(temp_model_config_path, "w") as f:
    yaml.dump(model_data, f)
print(f"Created temporary model config: {temp_model_config_path}")

# Slice prompts.json to extract California Schools prompts.
with open(prompts_path, "r") as f:
    all_prompts = json.load(f)

california_prompts = [
    p for p in all_prompts if p.get("db_id") == "california_schools"
]
sliced_prompts = california_prompts[:3]

with open(temp_prompts_path, "w") as f:
    json.dump(sliced_prompts, f, indent=2)
print(f"Created temporary prompts slice: {temp_prompts_path}")

# Generate the temporary run configuration with absolute paths.
run_config_content = f"""
dataset_config: {temp_prompts_path}

database_configs:
 - {os.path.join(base_path, 'datasets/bird/db_configs/sqlite.yaml')}

dialects:
 - sqlite
query_types:
 - dql
dataset_format: bird-standard-format

model_config: {temp_model_config_path}
prompt_generator: 'NOOPGenerator'

scorers:
  set_match: null
  exact_match: null

reporting:
  csv: {{}}
"""

with open(temp_run_config_path, "w") as f:
    f.write(run_config_content.strip())
print(f"Created temporary run config: {temp_run_config_path}")

## Step 7: Run EvalBench evaluation pipeline

We can now trigger the EvalBench evaluation run using the updated configuration. This executes query generation on your deployed agent and performs BigQuery execution and correctness scoring.

In [ ]:
import os
# Change to the evalbench root directory
%cd /content/evalbench

# Run the simplified execution command without the Python protobuf override
! .venv/bin/python3 evalbench/evalbench.py \
    --experiment_config="datasets/bird/test_agent_runtime_run_config_temp.yaml"